# 06b. LayerNorm: один токен за раз

Лекция: [LayerNorm](../../notes/interview-prep/08h-layernorm.md).

Сначала прочитай разделы 1–4 лекции и сделай только A. B — следующий небольшой шаг, C — готовый опыт для совместного разбора. Attention не переписываем.

Цель: понять, какие числа нормализуются вместе. Выводить производную LayerNorm сейчас не нужно.

In [1]:
import torch
from torch import nn

torch.set_printoptions(precision=4, sci_mode=False)
eps = 1e-5


## A. Один вектор токена

Вектор [1,3,5]. Его среднее — 3. Вычтем среднее, затем разделим на корень из дисперсии плюс eps.

Заполни три строки. Подсказки: `token.mean()`, `token - mean`, `centered.square().mean()`. Здесь усредняем квадраты отклонений с делением на D.

In [2]:
token = torch.tensor([1., 3., 5.])
mean = torch.mean(token)
centered = token - mean
variance = centered.square().mean()

assert isinstance(variance, torch.Tensor), 'Сначала заполни три строки выше'
normalized = centered / torch.sqrt(variance + eps)
reference = nn.LayerNorm(3, eps=eps, elementwise_affine=False)(token)
torch.testing.assert_close(mean, torch.tensor(3.))
torch.testing.assert_close(variance, torch.tensor(8 / 3))
torch.testing.assert_close(normalized, reference)
print('Среднее:', mean)
print('Отклонения:', centered)
print('Дисперсия:', variance)
print('Нормализованный вектор:', normalized)


Среднее: tensor(3.)
Отклонения: tensor([-2.,  0.,  2.])
Дисперсия: tensor(2.6667)
Нормализованный вектор: tensor([-1.2247,  0.0000,  1.2247])


### Остановка A

1. Какие числа мы усреднили: признаки одного токена или разные токены?
2. Изменилась ли длина вектора после нормализации?

**Мои ответы:**

1. признаки одного токена
2. нет, не изменилась

Сохрани ноутбук и пришли на проверку. На сегодня не нужно торопиться к C.

## B. Та же операция для (B,T,D)

Теперь B=2, T=2, D=3. `mean(dim=-1, keepdim=True)` вычисляет среднее по трём признакам каждого токена и оставляет форму (2,2,1).

Первый токен [1,3,5] и второй [11,13,15] различаются общим сдвигом. После вычитания своего среднего они совпадут.

In [3]:
x = torch.tensor([[[1., 3., 5.], [11., 13., 15.]],
                  [[2., 2., 2.], [-1., 0., 1.]]])
means = x.mean(dim=-1, keepdim=True)
centered_batch = x - means
variances = centered_batch.square().mean(dim=-1, keepdim=True)
manual = centered_batch / torch.sqrt(variances + eps)

norm = nn.LayerNorm(3, eps=eps, elementwise_affine=False)
assert isinstance(norm, nn.LayerNorm), 'Создай слой нормализации'
y = norm(x)
assert means.shape == (2, 2, 1)
assert y.shape == x.shape
torch.testing.assert_close(y, manual)
torch.testing.assert_close(y[0, 0], y[0, 1])
torch.testing.assert_close(y[1, 0], torch.zeros(3))

# Меняем другой токен и проверяем независимость первого.
changed = x.clone()
changed[0, 1] = torch.tensor([100., -50., 8.])
torch.testing.assert_close(norm(changed)[0, 0], y[0, 0])
print('Средние:', means)
print('После LayerNorm:', y)


Средние: tensor([[[ 3.],
         [13.]],

        [[ 2.],
         [ 0.]]])
После LayerNorm: tensor([[[-1.2247,  0.0000,  1.2247],
         [-1.2247,  0.0000,  1.2247]],

        [[ 0.0000,  0.0000,  0.0000],
         [-1.2247,  0.0000,  1.2247]]])


### Остановка B

1. Почему средних четыре, а не одно на весь x?
2. Почему изменение второго токена не изменило нормализованный первый?
3. Что произошло с [2,2,2]? Почему не получилось деления на ноль?

**Мои ответы:**

1. Потому что среднее считается для одной последовательности. У нас 2 батча по 2 последовательности. Для каждой по одному среднему
2. Потому что нормализация считется для каждого токена. Первый токен не менялся - нормализация не менялась. А для второго изменится
3. eps защитило от деления на ноль.

**Разбор ментора (исходные ответы сохранены):**

1. Здесь один batch из двух последовательностей, по два токена в каждой. Среднее считается по трём признакам каждого токена, не по последовательности: 2*2=4 средних, форма (2,2,1).
2. Верно: статистики разных токенов независимы. В данном примере нормализованный второй токен изменился; вообще изменение входа не всегда меняет нормализованный выход (например, общий сдвиг всех координат убирается вычитанием среднего).
3. Верно про eps. Дополнение: среднее [2,2,2] равно 2, отклонения и дисперсия равны нулю. Получаем 0/sqrt(eps)=0 для каждой координаты. В части B affine отключён, поэтому выход [0,0,0].


## C. Обучаемые масштаб и сдвиг — готовый опыт

В обычном LayerNorm есть gamma (weight) и beta (bias). Сначала gamma=1, beta=0. Вручную поменяем их, чтобы увидеть смысл: обучение здесь НЕ запускаем.

`no_grad` нужен только для ручной записи параметров. Статистики по-прежнему считаются из входа.

In [4]:
affine_norm = nn.LayerNorm(3, eps=eps)
assert sum(p.numel() for p in affine_norm.parameters()) == 6
torch.testing.assert_close(affine_norm(x), manual)

with torch.no_grad():
    affine_norm.weight.copy_(torch.tensor([2., 1., 0.5]))
    affine_norm.bias.copy_(torch.tensor([10., 0., -1.]))

affine_output = affine_norm(x)
expected = manual * affine_norm.weight + affine_norm.bias
torch.testing.assert_close(affine_output, expected)
print('weight:', affine_norm.weight)
print('bias:', affine_norm.bias)
print('Выход:', affine_output)
print('Средние после affine (уже не обязательно нули):', affine_output.mean(dim=-1))


weight: Parameter containing:
tensor([2.0000, 1.0000, 0.5000], requires_grad=True)
bias: Parameter containing:
tensor([10.,  0., -1.], requires_grad=True)
Выход: tensor([[[ 7.5505,  0.0000, -0.3876],
         [ 7.5505,  0.0000, -0.3876]],

        [[10.0000,  0.0000, -1.0000],
         [ 7.5505,  0.0000, -0.3876]]], grad_fn=<NativeLayerNormBackward0>)
Средние после affine (уже не обязательно нули): tensor([[2.3876, 2.3876],
        [3.0000, 2.3876]], grad_fn=<MeanBackward1>)


In [5]:
affine_output

tensor([[[ 7.5505,  0.0000, -0.3876],
         [ 7.5505,  0.0000, -0.3876]],

        [[10.0000,  0.0000, -1.0000],
         [ 7.5505,  0.0000, -0.3876]]], grad_fn=<NativeLayerNormBackward0>)

### После совместного разбора C

1. Что обучается: mean/variance или gamma/beta?
2. Для входа (8,20,64) и nn.LayerNorm(64) назови форму выхода и количество параметров по умолчанию.

**Мои ответы:**

1. gamma/beta
2. так и будет (8,20,64), параметров 64+gamma+beta

Дальше: MLP и сборка pre-norm Transformer-блока из готовых компонентов. Перед проверкой выполненной практики: Restart Kernel → Run All → сохранить. Не выполняй все ячейки до заполнения TODO.

**Разбор ментора (исходные ответы сохранены):**

1. Верно: обучаются gamma и beta. Mean и variance вычисляются из текущего входа, а не хранятся как обучаемые параметры.
2. Форма (8,20,64) верная. Параметров 128: gamma — вектор из 64 чисел, beta — ещё 64 числа. Не 64 плюс два скаляра. Эти векторы общие для всех 8*20 токенов данного слоя; у каждого токена свои среднее и дисперсия. Формула для координаты i: y_i = gamma_i * (x_i - mean) / sqrt(variance + eps) + beta_i. Здесь покоординатное умножение, не матричное.

Код и сохранённые результаты просмотрены без запуска ноутбука или проверок. В C параметры менялись вручную внутри no_grad; обучения через backward/step в этом опыте нет.
